# D3-01 Day 2 recap and Day 3 readiness checks

Run this notebook in the `bw` environment. The goal is to confirm that the Day 2 regionalised LCIA workflow is still in place before we move into the Day 3 extensions.


## Learning goals

After this notebook, you should be able to:

- verify that the Day 2 project, `edges`, and key regionalised methods are available
- restate what changed from conventional LCIA to exchange-based regionalised LCIA
- rerun a compact `EdgeLCIA` calculation and inspect the matched CF table behind the score
- identify which Day 2 outputs Day 3 will reuse for parameterised methods and inventory-flow assessment


## Background references

- Sacchi, R., Menacho, A. H., Seitfudem, G., Agez, M., Schlesinger-Martinat, J., Koyamparambath, A., Saldivar, J. S., Loubet, P., & Bauer, C. (2025). Contextual LCIA without the overhead: an exchange-based framework for flexible impact assessment. *The International Journal of Life Cycle Assessment, 30*(12), 3087-3101. https://doi.org/10.1007/s11367-025-02551-7
- Seitfudem, G., Berger, M., Schmied, H. M., & Boulay, A. (2025). The updated and improved method for water scarcity impact assessment in LCA, AWARE2.0. *Journal of Industrial Ecology, 29*(3), 891-907. https://doi.org/10.1111/jiec.70023
- Day 2 notebooks `D2-03`, `D2-05`, and `D2-06` in this repository


## 1) Confirm imports, project state, and method availability

Day 3 continues in the same Brightway project as Day 2. We start with a quick check that the core packages import correctly and that a few representative regionalised methods are available.


In [ ]:
import importlib
import importlib.metadata as md

import bw2data as bd
import pandas as pd
from IPython.display import display
from edges import EdgeLCIA, get_available_methods

bd.projects.set_current('barcelona-rlcia-2026')
database_names = sorted(bd.databases)
available_methods = get_available_methods()
method_examples = [
    method for method in available_methods
    if method[0] in {'AWARE 2.0', 'ImpactWorld+ 2.1'}
][:8]

print('Current project:', bd.projects.current)
print('Number of databases:', len(database_names))
print('Number of Brightway methods:', len(bd.methods))
print('Number of edges methods:', len(available_methods))
print()
print('Databases in the current project:')
for name in database_names:
    print('-', name)

bafu_candidates = [name for name in database_names if name.startswith('bafu')]
print()
print('BAFU database candidates:', bafu_candidates)

pd.DataFrame({'example regionalised methods': method_examples})


## 2) Recall the Day 2 workflow

Day 1 already gave us the main LCA structure: choose a demand, build the inventory, apply characterization, and interpret the score.

Day 2 kept that overall flow, but changed the characterization logic:

1. choose an activity and a regionalised method
2. run `lci()` to build the inventory
3. map exchanges and locations
4. evaluate the exchange-level CFs
5. run `lcia()` and inspect the matched exchanges with `generate_cf_table()`

So the Day 3 starting point is not a new project or a new inventory formalism. It is the same Brightway project plus a clearer view of which exchange-level objects we want to inspect more closely.


## Checkpoint 1

In your own words, answer the following two questions:

- What stays the same between a Day 1 `LCA` run and a Day 2 `EdgeLCIA` run?
- Why can the same biosphere flow receive different CFs in `edges`?


## 3) Re-run one compact regionalised LCIA example

We reuse the `Irrigating` activity from `D2-06` with `AWARE 2.0`. The point is not the exact score, but to verify that you can still identify a real BAFU activity, pick a regionalised method, run `EdgeLCIA`, and inspect the matched exchanges behind the result.


In [ ]:
db_name = 'bafu'
db = bd.Database(db_name)

activity = next(
    act for act in db
    if act['name'] == 'Irrigating'
    and act.get('location') == 'CH'
)
aware_method = ('AWARE 2.0', 'Country', 'all', 'yearly')
assert aware_method in available_methods

lca = EdgeLCIA({activity: 1}, aware_method)
lca.lci()
lca.apply_strategies()
lca.evaluate_cfs()
lca.lcia()

cf_table = lca.generate_cf_table(include_unmatched=False)
activity_matches = cf_table[
    (cf_table['consumer name'] == activity['name'])
    & (cf_table['consumer reference product'] == activity['reference product'])
    & (cf_table['consumer location'] == activity['location'])
    & (cf_table['CF'].notna())
][['supplier name', 'supplier categories', 'consumer location', 'amount', 'CF', 'impact']].copy()

summary = pd.Series(
    {
        'activity': activity['name'],
        'location': activity['location'],
        'method': ' / '.join(aware_method),
        'matched exchanges on focal activity': len(activity_matches),
        'total score': float(lca.score),
    },
    name='Recap run',
)

summary.to_frame('value')
activity_matches.head(10)


## 4) What Day 3 adds with `edges`

The recap run above still used one fixed regionalised method and one fixed way of evaluating characterization factors. Day 3 goes one step further: we keep the `edges` logic from Day 2, but make the characterization step itself more flexible.

Today, we use `edges` in three directions:

- parameterise the **CF values** themselves, so one method can hold several CF sets
- parameterise the **evaluation of CFs**, so matched exchanges can be re-evaluated under different assumptions
- characterise **intermediate product flows**, meaning technosphere-to-technosphere exchanges and not only biosphere flows

So the Day 3 extension is still about exchange-level characterization, but with more control over both the factors we use and the exchanges we decide to characterize.
